# Image Generation Walkthrough

This notebook is the interactive route through the image-generation companion script. The script remains the source of truth; the notebook checks dependencies, inspects committed reference artifacts, and gives a guarded quick run for local experimentation.

Read the cells as a public-repo scaffold: they separate lightweight inspection from optional training, show which files are evidence, and keep generated runs out of version control by default.

## Reader Preflight

Before running this notebook as public companion code:

- Start with the dependency check or smoke path. Real training, generation, or evaluation cells are usually guarded by flags such as `RUN_* = False`.
- Confirm dataset and model access, license or terms, and local paths before enabling external downloads or long runs.
- Keep secrets out of notebook cells. If a token is required, load it from the environment or `.env`, and keep `.env` secrets-only.
- Treat printed paths and saved JSON, CSV, and PNG artifacts as the evidence record. Rerun from a clean kernel before reporting results.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys


def find_chapter_dir(script_name: str, chapter_name: str) -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        for candidate in (base, base / chapter_name, base / "code" / chapter_name):
            if (candidate / script_name).exists():
                return candidate.resolve()
    raise FileNotFoundError(f"Could not locate {script_name}")


CHAPTER_DIR = find_chapter_dir("image_generation_experiments.py", "chapter_image_generation")
SCRIPT = CHAPTER_DIR / "image_generation_experiments.py"


def run_script(*args: object) -> subprocess.CompletedProcess[str]:
    cmd = [sys.executable, str(SCRIPT), *map(str, args)]
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, cwd=CHAPTER_DIR, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    result.check_returncode()
    return result


print(f"Chapter directory: {CHAPTER_DIR}")

## Dependency Check

The check reports optional packages without requiring them to be installed. Missing optional packages are expected on a lightweight reader machine; install the `image-generation` dependency group before running the quick experiment cells.

In [ ]:
run_script("--check-deps", "--allow-missing-deps")


## Reference Artifact Inventory

The public repo includes compact CSV and JSON records used by the chapter tables. Treat these files as table inputs and reproducibility records, not as notebook output that must be regenerated. PNG grids are generated by full experiment runs and rendered in the book.

In [ ]:
reference_dir = CHAPTER_DIR / "reference_artifacts"
for path in sorted(reference_dir.glob("*")):
    if path.is_file():
        print(f"{path.name}: {path.stat().st_size:,} bytes")

values_path = reference_dir / "imagegen-reference-result-values.json"
if values_path.exists():
    values = json.loads(values_path.read_text(encoding="utf-8"))
    print("
Result value keys:")
    for key in sorted(values):
        print("-", key)


## Quick Synthetic Run

The quick run trains tiny classroom models on generated 32 by 32 shape images. It is still an ML run, so leave it disabled until the local kernel has PyTorch, NumPy, and Pillow available. When enabled, use it to verify the pipeline and artifact contract, not to make an image-quality claim.

In [ ]:
RUN_QUICK = False
quick_dir = CHAPTER_DIR / "artifacts" / "image_generation_quick"

if RUN_QUICK:
    run_script("--mode", "quick", "--output-dir", quick_dir)
else:
    print("Set RUN_QUICK = True after installing the image-generation dependencies.")
    print(f"Quick artifacts will be written under {quick_dir}")


## Inspect Quick Outputs

After a quick run, inspect the written artifact names before comparing numbers. Look for metadata, CSV or JSON summaries, and generated image grids. The quick mode is a pipeline check; surprising metrics usually mean the toy setup or local environment needs inspection, not that a model result is reportable.

In [ ]:
if quick_dir.exists():
    for path in sorted(quick_dir.iterdir()):
        print(path.name)
else:
    print(f"No quick output directory yet: {quick_dir}")


## Reference Run Command

Use a GPU machine for the reference run. Review the output directory, device choice, seed, runtime, and memory fields before launching. Keep generated `runs/` directories out of version control and copy only reviewed compact artifacts when updating the book evidence.

In [ ]:
reference_command = [
    sys.executable,
    str(SCRIPT),
    "--mode", "reference",
    "--device", "auto",
    "--output-dir", "runs/image-generation-reference",
]
print(" ".join(reference_command))


## What To Report

For a real run, record the exact command, package versions, seed, device, runtime, memory use, and output artifact names. Compare the compact CSV and JSON tables rather than screenshots. Do not commit generated samples, checkpoints, prompts, or full run directories; keep only reviewed reference artifacts that support the chapter text.

## Reader Report Checklist

Before using results from this notebook in a report or downstream example, record:

- The notebook name, companion script, package versions, device, seed, and any guarded flags you enabled.
- The dataset or sample-data source, license or terms, split policy, and any preprocessing or synthetic fallback used.
- The exact artifact files that support the result, preferably saved JSON, CSV, or PNG files rather than transient cell output.
- The validation evidence used for model or configuration selection, and whether any final-test cell was run exactly once.
- The main limitation of the run, such as tiny synthetic data, missing optional dependencies, short training budget, or unavailable model access.